# Experiment 5A: DigiSteel v4 — DAFE (Defect-Aware Feature Enhancement)

**Architecture:** YOLOv11n + DAFE at P2 and P3
**DAFE:** Dual-branch module — Sobel edge detection (crazing/scratches) + local variance (pitting/scale)
**Baseline:** Fresh baseline YOLOv11n = 78.8% mAP@0.5 (test)
**Goal:** 80%+ mAP@0.5 with real architectural improvement

Same optimized recipe as fresh baseline — only change is DAFE module.

In [ ]:
# Cell 1: Setup
import sys
from pathlib import Path

# Add project root to Python path so digisteel package is importable
ROOT = Path(r"D:/DigiSteel-Yolo/DigiSteel-YOLO")
sys.path.insert(0, str(ROOT))

from ultralytics import YOLO
import ultralytics.nn.tasks as tasks
from digisteel.modules import DAFE
from digisteel.engine.trainer import register_custom_modules
import json
import time
import torch

# Register custom modules BEFORE loading YAML
register_custom_modules()
print('Custom modules registered: DAFE, CoordAttention, GhostConv, WFCA, EMA')

# Paths
DATA_YAML = ROOT / "configs" / "data" / "neu_det.yaml"
MODEL_YAML = ROOT / "configs" / "models" / "digisteel.yaml"
RUNS_DIR = ROOT / "runs" / "detect"
RUN_NAME = "exp_5a_digisteel_v4_dafe"
BEST_PT = RUNS_DIR / RUN_NAME / "weights" / "best.pt"
EVALS_DIR = ROOT / "evals"
BASELINE_RESULTS = EVALS_DIR / "week4_4a_results.json"

print(f'Model config: {MODEL_YAML}')
print(f'Data config: {DATA_YAML}')
print(f'Output: {RUNS_DIR / RUN_NAME}')

# Verify paths
assert MODEL_YAML.exists(), f"Model YAML not found: {MODEL_YAML}"
assert DATA_YAML.exists(), f"Data YAML not found: {DATA_YAML}"

# Load baseline results for comparison
if BASELINE_RESULTS.exists():
    with open(BASELINE_RESULTS) as f:
        baseline = json.load(f)
    print(f'\nBaseline mAP@0.5: {baseline["map50"]*100:.1f}%')
    print(f'Baseline mAP@0.5:0.95: {baseline["map50_95"]*100:.1f}%')
else:
    print('Warning: Baseline results not found for comparison')
    baseline = None


## Build Model

In [ ]:
# Cell 2: Build DigiSteel v4 model from YAML + load pretrained weights
print('Building DigiSteel v4 (YOLOv11n + DAFE)...')
model = YOLO(str(MODEL_YAML))

# Load pretrained YOLOv11n weights (transfer learning)
model.load('yolo11n.pt')

# Print model info
info = model.info()
print(f'\nModel built successfully.')
print(f'Architecture: YOLOv11n + DAFE at P2 and P3')
print(f'Pretrained weights: yolo11n.pt (transfer learning)')

## Training

In [ ]:
# Cell 3: Train with optimized recipe (same as fresh baseline)
import time

base_overrides = {
    'data': str(DATA_YAML),
    'task': 'detect',
    'epochs': 600,
    'patience': 150,
    'batch': 16,
    'imgsz': 800,
    'device': 0,
    'optimizer': 'AdamW',
    'lr0': 0.001,
    'lrf': 0.01,
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 5,
    'warmup_momentum': 0.8,
    'warmup_bias_lr': 0.1,
    'mosaic': 0.0,
    'mixup': 0.15,
    'copy_paste': 0.1,
    'copy_paste_mode': 'flip',
    'degrees': 10.0,
    'translate': 0.2,
    'scale': 0.6,
    'shear': 5.0,
    'perspective': 0.0,
    'flipud': 0.5,
    'fliplr': 0.5,
    'hsv_h': 0.015,
    'hsv_s': 0.7,
    'hsv_v': 0.4,
    'erasing': 0.4,
    'cos_lr': True,
    'deterministic': True,
    'close_mosaic': 10,
    'amp': True,
    'seed': 42,
    'workers': 8,
    'save': True,
    'save_period': 50,
    'plots': True,
    'project': str(RUNS_DIR),
    'name': RUN_NAME,
    'exist_ok': True,
    'verbose': True,
}

print('=' * 60)
print('TRAINING CONFIGURATION')
print('=' * 60)
print(f'Architecture:  YOLOv11n + DAFE (P2 + P3)')
print(f'Weights:       Pretrained (transfer learning)')
print(f'Dataset:       {DATA_YAML}')
print(f'Epochs:        {base_overrides["epochs"]}')
print(f'Patience:      {base_overrides["patience"]}')
print(f'Batch:         {base_overrides["batch"]}')
print(f'Image size:    {base_overrides["imgsz"]}')
print(f'Optimizer:     {base_overrides["optimizer"]}')
print(f'Mosaic:        {base_overrides["mosaic"]}')
print(f'Mixup:         {base_overrides["mixup"]}')
print(f'Copy-paste:    {base_overrides["copy_paste"]} ({base_overrides["copy_paste_mode"]})')
print(f'Output:        {RUNS_DIR / RUN_NAME}')
print('=' * 60)
print()

training_start = time.time()
copy_paste_used = base_overrides['copy_paste']

try:
    print(f'Starting training with copy_paste={copy_paste_used}...')
    results = model.train(**base_overrides)
except Exception as e:
    error_msg = str(e).lower()
    if 'copy_paste' in error_msg or 'segment' in error_msg or 'mask' in error_msg:
        print(f'\ncopy_paste={copy_paste_used} failed. Retrying without it...')
        print(f'Error: {e}')
        base_overrides.pop('copy_paste', None)
        copy_paste_used = 0.0
        model = YOLO(str(MODEL_YAML))
        model.load('yolo11n.pt')
        results = model.train(**base_overrides)
    else:
        raise

training_time = time.time() - training_start

print(f'\nTraining complete in {training_time/3600:.1f} hours')
print(f'Copy-paste used: {copy_paste_used}')
print(f'Best weights: {BEST_PT}')

## Evaluate on Test Set

In [ ]:
# Cell 4: Evaluate on TEST set
print('Loading best weights for evaluation...')
best_model = YOLO(str(BEST_PT))

print('Running evaluation on TEST set...')
val_results = best_model.val(
    data=str(DATA_YAML),
    split='test',
    imgsz=800,
    batch=16,
    device=0,
    plots=True,
    verbose=True,
)

# Extract metrics
map50 = float(val_results.box.map50)
map50_95 = float(val_results.box.map)
precision = float(val_results.box.mp)
recall = float(val_results.box.mr)

# Per-class AP@0.5
class_names = val_results.names
per_class_ap50 = {}
for cls_id, cls_name in class_names.items():
    if cls_id < len(val_results.box.ap50):
        per_class_ap50[cls_name] = float(val_results.box.ap50[cls_id])
    else:
        per_class_ap50[cls_name] = 0.0

print(f'\n{"="*60}')
print('TEST SET EVALUATION RESULTS')
print(f'{"="*60}')
print(f'mAP@0.5:      {map50:.4f} ({map50*100:.1f}%)')
print(f'mAP@0.5:0.95: {map50_95:.4f} ({map50_95*100:.1f}%)')
print(f'Precision:    {precision:.4f}')
print(f'Recall:       {recall:.4f}')

# Compare with baseline
if baseline:
    print(f'\n--- vs Baseline ---')
    print(f'mAP@0.5:      {baseline["map50"]*100:.1f}% -> {map50*100:.1f}% ({(map50-baseline["map50"])*100:+.1f}%)')
    print(f'mAP@0.5:0.95: {baseline["map50_95"]*100:.1f}% -> {map50_95*100:.1f}% ({(map50_95-baseline["map50_95"])*100:+.1f}%)')
    print(f'Precision:    {baseline["precision"]*100:.1f}% -> {precision*100:.1f}% ({(precision-baseline["precision"])*100:+.1f}%)')
    print(f'Recall:       {baseline["recall"]*100:.1f}% -> {recall*100:.1f}% ({(recall-baseline["recall"])*100:+.1f}%)')

print(f'\nPer-class AP@0.5:')
print(f'{"-"*40}')
for cls_name, ap in per_class_ap50.items():
    bl_ap = baseline['per_class_ap50'].get(cls_name, 0) if baseline else 0
    diff = ap - bl_ap
    sign = '+' if diff > 0 else ''
    print(f'  {cls_name:<20} {ap*100:.1f}% (baseline: {bl_ap*100:.1f}%, {sign}{diff*100:.1f}%)')
print(f'{"="*60}')

## FPS Benchmark

In [ ]:
# Cell 5: FPS measurement
model = YOLO(str(BEST_PT))
test_dir = ROOT / 'datasets' / 'NEU-DET' / 'yolo' / 'images' / 'test'
test_images = list(test_dir.glob('*.jpg'))
print(f'Test images: {len(test_images)}')

# Warmup
for img in test_images[:5]:
    model(str(img), verbose=False)

# Measure FPS
start = time.time()
for img in test_images:
    model(str(img), verbose=False)
fps_time = time.time() - start
fps = len(test_images) / fps_time

print(f'\nFPS: {fps:.1f} ({fps_time:.1f}s for {len(test_images)} images)')

# Baseline FPS for comparison
if baseline and 'fps' in baseline:
    bl_fps = baseline['fps'].get('no_tta', 0)
    print(f'Baseline FPS: {bl_fps:.1f}')
    print(f'DAFE overhead: {bl_fps/fps:.2f}x')

## Save Results

In [ ]:
# Cell 6: Save results to JSON
metrics = {
    'experiment': '5A_digisteel_v4_dafe',
    'architecture': 'YOLOv11n + DAFE (P2 + P3)',
    'model': 'yolo11n',
    'weights': str(BEST_PT),
    'map50': map50,
    'map50_95': map50_95,
    'precision': precision,
    'recall': recall,
    'per_class_ap50': per_class_ap50,
    'training_time_hours': round(training_time / 3600, 2),
    'copy_paste_used': copy_paste_used,
    'fps': fps,
    'hyperparameters': {
        'epochs': 600,
        'patience': 150,
        'batch': 16,
        'imgsz': 800,
        'optimizer': 'AdamW',
        'lr0': 0.001,
        'mosaic': 0.0,
        'mixup': 0.15,
        'copy_paste': copy_paste_used,
        'cos_lr': True,
    },
}

# Add baseline comparison if available
if baseline:
    metrics['vs_baseline'] = {
        'map50_delta': map50 - baseline['map50'],
        'map50_95_delta': map50_95 - baseline['map50_95'],
        'precision_delta': precision - baseline['precision'],
        'recall_delta': recall - baseline['recall'],
        'per_class_delta': {
            cls: per_class_ap50[cls] - baseline['per_class_ap50'].get(cls, 0)
            for cls in per_class_ap50
        },
    }

out_path = EVALS_DIR / 'exp_5a_digisteel_v4_dafe.json'
EVALS_DIR.mkdir(parents=True, exist_ok=True)
with open(out_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'Results saved to: {out_path}')

# Final summary
print(f'\n{"="*60}')
print('FINAL SUMMARY')
print(f'{"="*60}')
print(f'Architecture: YOLOv11n + DAFE (P2 + P3)')
print(f'mAP@0.5:      {map50*100:.1f}%')
if baseline:
    delta = (map50 - baseline['map50']) * 100
    sign = '+' if delta > 0 else ''
    print(f'vs Baseline:  {sign}{delta:.1f}% (baseline: {baseline["map50"]*100:.1f}%)')
print(f'mAP@0.5:0.95: {map50_95*100:.1f}%')
print(f'Precision:    {precision*100:.1f}%')
print(f'Recall:       {recall*100:.1f}%')
print(f'FPS:          {fps:.1f}')
if map50 >= 0.80:
    print(f'\n*** 80% TARGET REACHED! ***')
else:
    gap = (0.80 - map50) * 100
    print(f'\nGap to 80%: {gap:.1f}%')
print(f'{"="*60}')